# Буldіршін — KazNERD сынағы (Colab)

Бұл дәптер **ноутбукті жүктемейді**: бәрі Google серверінде орындалады.

**Не істейді:** KazNERD моделін (`yeshpanovrustem/xlm-roberta-large-kaznerd`) біздің
XVIII ғасыр жыр мәтіндерінде сынайды — есім мен жер атын қаншалық дұрыс табатынын өлшейді.

**Алдымен GPU қосыңыз:** `Runtime → Change runtime type → Hardware accelerator → T4 GPU`
(тегін). GPU-сыз да жұмыс істейді, тек баяуырақ.

---
### 1-қадам. Кітапханаларды орнату

In [ ]:
# torch-ты ЖАҢАРТПАҢЫЗ! Colab-та torch пен torchvision жұптасып орнатылған.
# Біреуін жаңартсаңыз: "RuntimeError: operator torchvision::nms does not exist"
!pip -q install -U transformers

import torch
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "жоқ (CPU режимі — жарайды)")

### 1а. Егер `torchvision::nms` қатесі шықса

Бұл — torch пен torchvision нұсқаларының сәйкессіздігі, мәтін тапсырмасына қатысы жоқ.
Төмендегі ұяшық тексеріп, қажет болса жөндейді.

In [ ]:
import torch, subprocess, sys
try:
    import torchvision, torchvision.ops
    print(f"✓ сәйкес: torch {torch.__version__} | torchvision {torchvision.__version__}")
except Exception as e:
    print("✗ сәйкессіздік:", type(e).__name__, e)
    print("\ntorchvision жойылады — мәтін тапсырмасына қажет емес.")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"])
    print("\n⚠ ЕНДІ: Runtime → Restart session, сосын осы ұяшықтан бастап қайта жүргізіңіз.")

### 2-қадам. Репозиторийді алу

Репозиторий жабық болса, төмендегі ұяшық қатемен аяқталады — сонда **3-қадамға**
өтіп, файлдарды қолмен жүктеңіз.

In [ ]:
REPO = "https://github.com/zakcination/Buldirshin.git"
BRANCH = "claude/kazakh-heroes-research-2mkf89"
# Жабық репозиторий үшін: REPO = "https://<USERNAME>:<TOKEN>@github.com/zakcination/Buldirshin.git"
!git clone --depth 1 -b $BRANCH $REPO 2>&1 | tail -2
%cd Buldirshin

### 3-қадам (баламалы). Файлдарды қолмен жүктеу

Клондау сәтсіз болса, `pipeline/eval/gold/` бумасындағы файлдарды осы жерге
жүктеңіз: `annotations.json` және үш `.txt` мәтін.

In [ ]:
# from google.colab import files
# import os
# os.makedirs("pipeline/eval/gold", exist_ok=True)
# up = files.upload()
# for name, data in up.items():
#     open(f"pipeline/eval/gold/{name}", "wb").write(data)
# print("жүктелді:", list(up))

### 4-қадам. Сынақты жүргізу

Модель ~2,2 ГБ — алғаш жүктеу бірнеше минут алады.

> **Скрипт `transformers.pipeline`-ды әдейі пайдаланбайды.** Себебі екеу:
> (1) оның импорты torchvision тізбегін тартады — жоғарыдағы қате содан шыққан;
> (2) `aggregation_strategy` баптауы нәтижені 18%-дан 79%-ға дейін өзгертеді, оны
> кітапханаға қалдырмай өзіміз бақылаған дұрыс. Субсөзді біріктіру қолмен жасалады.

In [ ]:
!python3 pipeline/eval/run_ner_eval.py

### 5-қадам. Агрегация тәсілдерін салыстыру

Бұл ұяшық жоғарыдағы ескертуді өз көзіңізбен көрсетеді.

In [ ]:
# Агрегация тәсілінің әсері — pipeline арқылы (torchvision дұрыс болса ғана жұмыс істейді).
# Қате шықса, бұл ұяшықты өткізіп жіберіңіз: негізгі нәтиже 4-қадамда алынды.
try:
    import json, re, torch
    from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
    M = "yeshpanovrustem/xlm-roberta-large-kaznerd"
    tok = AutoTokenizer.from_pretrained(M)
    mdl = AutoModelForTokenClassification.from_pretrained(M)
    gold = json.load(open("pipeline/eval/gold/annotations.json"))
    def norm(s): return re.sub(r"[^\w]", "", s.lower())
    def match(g, p):
        g, p = norm(g), norm(p)
        if len(g) >= 4 and g in p: return True
        n = min(len(g), len(p))
        return (g == p) if n < 4 else g[:n] == p[:n]
    for strat in ("simple", "max", "first", "average"):
        ner = pipeline("ner", model=mdl, tokenizer=tok, aggregation_strategy=strat,
                       device=0 if torch.cuda.is_available() else -1)
        tp = fn = 0
        for tid, ann in gold.items():
            if tid.startswith("_"): continue
            cand = set()
            for line in open(f"pipeline/eval/gold/{tid}.txt"):
                if line.strip():
                    for e in ner(line.strip()):
                        if "PERSON" in e["entity_group"].upper():
                            cand.add(e["word"].strip())
            g = ann.get("PER", [])
            f = [x for x in g if any(match(x, c) for c in cand)]
            tp += len(f); fn += len(g) - len(f)
        print(f"aggregation={strat:<9} PER Recall = {tp/(tp+fn):.0%}  ({tp}/{tp+fn})")
except Exception as e:
    print("pipeline қолжетімсіз:", type(e).__name__, e)
    print("Бұл қалыпты жағдай — негізгі сынақ 4-қадамда pipeline-сыз орындалды.")

### 6-қадам. Өз мәтініңізде сынау

Кез келген жыр не аңыз мәтінін қойып, модель не табатынын көріңіз.

In [ ]:
import sys, torch
sys.path.insert(0, "pipeline/eval")
from run_ner_eval import tag_line, MODEL
from transformers import AutoTokenizer, AutoModelForTokenClassification

tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL).eval()

TEXT = """Қаракерей Қабанбай,
Қанжығалы Бөгенбай,
Қаз дауысты Қазыбек,
Шақшақұлы Жәнібек -
Ормандай көп Орта жүз,
Сондан шыққан төрт тірек."""

for line in TEXT.strip().splitlines():
    ents = tag_line(line.strip(), tok, mdl, torch)
    if ents:
        print(line.strip())
        for w, lab in ents:
            print(f"     → {lab:<12} «{w}»")

---

## Нәтижені қалай оқу керек

| Өлшем | Күтілетін мән | Мағынасы |
|---|---|---|
| PER Recall | ~92% | Есімдерді табады — кандидат тізбесі ретінде жарайды |
| PER Precision | ~50–60% | Әр екінші болжам артық — **адам сүзгісі міндетті** |
| LOC Recall | 40–100% | Көне топонимде әлсіз (жаңалық мәтінінде оқытылған) |

**Басты ереже:** модель өндірген тізім — **C деңгейлі материал**, яғни
тексерілетін тұжырым тізбесі. Дәлел емес. Толығы: `ADISTEME.md`.